# 01 — Getting to know and cleaning the MMDEC data

**Project:** Maritime Traffic & Environmental Performance Analytics  
Before asking an analytical question, the first task is to understand what was recorded, what the fields mean, how the six files fit together, and whether the values are reliable enough to use.

This notebook documents that process. It covers the six supplied files, their columns and data types, missing values, repeated records, code values, units, and basic physical or geographic checks. It then applies and verifies a set of cleaning rules. It does not yet try to explain traffic patterns or build a model.

## Approach

The review starts with the file structure and column types, followed by actual values and missingness. Each check considers what the field represents, what values are expected, and what evidence would justify a change. Original files remain unchanged; cleaned copies are written to `data/processed/`. File relationships are described in `docs/data_inventory.md`, and cleaning results are recorded in `docs/cleaning_log.md`.

Run the notebook from top to bottom. The code is grouped so a reader can follow the observation, the reason for checking it, and the resulting decision.

In [1]:
# Import filesystem and table tools used to inspect local files without changing them.
from pathlib import Path
import json
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import pyarrow.compute as pc
from IPython.display import display

# Find the project root whether Jupyter starts in the root or notebooks/ folder.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

# List the six expected local files so a missing/renamed input is caught immediately.
FILES = {
    "AIS_POS": ROOT / "Dataset_AIS_POS.parquet",
    "AIS_SPEC": ROOT / "Dataset_AIS_SPEC.parquet",
    "AIS_ShipTypes": ROOT / "Dataset_AIS_ShipTypes.csv",
    "ERA5": ROOT / "Dataset_ERA5.parquet",
    "CMEMS_PHY": ROOT / "Dataset_CMEMS_PHY.parquet",
    "CMEMS_WAV": ROOT / "Dataset_CMEMS_WAV.parquet",
}
missing_files = [name for name, path in FILES.items() if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Expected local input files are missing: {missing_files}")
print(f"Found all {len(FILES)} local project files under {ROOT}")

Found all 6 local project files under /Users/midhunpraj/Documents/Maritime Traffic & Environmental Performance Analytics


## 1. Identify the six files and inspect how their columns are stored

The file contents are checked rather than inferred from their names. Parquet schemas show each column's stored type without loading every observation; the ship-type CSV is read using its semicolon separator. These storage types are then compared with the field descriptions: an integer might be a vessel identifier or a code, while a decimal might represent a physical measurement. This distinction matters before calculating summaries or changing types.

In [2]:
# Create one inventory row per input file with file size and stored row count.
inventory_rows = []
# Keep one ParquetFile handle per Parquet dataset for the subsequent audits.
PARQUET = {}
for name, path in FILES.items():
    # Read the CSV header and infer its fields using the source's semicolon delimiter.
    if path.suffix == ".csv":
        lookup = pd.read_csv(path, sep=";")
        inventory_rows.append({"dataset": name, "file": path.name, "bytes": path.stat().st_size,
                               "rows": len(lookup), "columns": len(lookup.columns), "format": "CSV ;"})
        continue
    # Read only Parquet footer metadata; this does not scan all data pages.
    parquet_file = pq.ParquetFile(path)
    PARQUET[name] = parquet_file
    inventory_rows.append({"dataset": name, "file": path.name, "bytes": path.stat().st_size,
                           "rows": parquet_file.metadata.num_rows,
                           "columns": len(parquet_file.schema_arrow), "format": "Parquet"})
# Display a compact six-file overview before inspecting individual fields.
display(pd.DataFrame(inventory_rows))

# Print every stored field and Arrow type so datatype assumptions are visible.
for name, parquet_file in PARQUET.items():
    print(f"\n{name}: {parquet_file.metadata.num_rows:,} rows")
    for field in parquet_file.schema_arrow:
        # Show timezone information explicitly for timestamp fields when present.
        print(f"  {field.name:<28} {field.type}")

# Display pandas-inferred CSV dtypes and a few rows for the type lookup.
print("\nAIS_ShipTypes CSV dtypes:")
display(lookup.dtypes.rename("dtype").to_frame())
display(lookup.head())

,dataset,file,bytes,rows,columns,format
0,AIS_POS,Dataset_AIS_POS.parquet,450935970,19014229,14,Parquet
1,AIS_SPEC,Dataset_AIS_SPEC.parquet,209957042,13558007,19,Parquet
2,AIS_ShipTypes,Dataset_AIS_ShipTypes.csv,9848,256,3,CSV ;
3,ERA5,Dataset_ERA5.parquet,13918170,537510,11,Parquet
4,CMEMS_PHY,Dataset_CMEMS_PHY.parquet,107608411,5007803,10,Parquet
5,CMEMS_WAV,Dataset_CMEMS_WAV.parquet,35550335,1670779,23,Parquet



AIS_POS: 19,014,229 rows
  Date                         timestamp[ns]
  Source                       string
  MessageType                  int32
  Mmsi                         uint32
  NavigationStatus             double
  Latitude                     double
  Longitude                    double
  PositionAccuracy             bool
  CourseOverGroundDegrees      float
  SpeedOverGround              float
  RateOfTurn                   double
  TrueHeadingDegrees           double
  chunk_folder                 string
  id_chunk                     list<element: string>

AIS_SPEC: 13,558,007 rows
  Date                         timestamp[ns]
  Source                       string
  MessageType                  int32
  Mmsi                         uint32
  ImoNumber                    uint32
  CallSign                     string
  VesselName                   string
  ShipType                     int32
  DimensionToBow               uint32
  DimensionToStern             uint32
  DimensionTo

,dtype
Code,int64
Type,object
Additional,object


,Code,Type,Additional
0,0,Not available,"Not available or no ship, default"
1,1,Other,Reserved for future use
2,2,Other,Reserved for future use
3,3,Other,Reserved for future use
4,4,Other,Reserved for future use


### Files identified

All six expected files are present. Two contain AIS reports: **AIS_POS** records vessel positions and movement-related fields, while **AIS_SPEC** contains vessel particulars and voyage information. **AIS_ShipTypes** is a lookup table that gives names to ship-type codes. The other three files are environmental grids: ERA5 weather, CMEMS_PHY ocean physics, and CMEMS_WAV wave conditions.

The AIS files describe reports from vessels; environmental files describe conditions at times and grid locations. Their records therefore need different checks. For example, repeated vessel reports and repeated grid cells have different meanings. The inventory document gives the file relationships and field descriptions in more detail.

### What the storage types tell us

The schema shows how a value is stored, but not automatically what it means. `Mmsi` identifies a vessel, so arithmetic on it would make no sense even though it is numeric. `MessageType`, `ShipType`, navigation status, position-fix type, and ETA parts are numbers used as codes. Latitude, longitude, speed, course, and environmental variables are measurements or coordinates.

The timestamps have no timezone label in these files. The recorded clock value is preserved, with a separate UTC-interpreted field added based on source conventions. This labels the time but does not shift it. Text fields can contain padding or AIS placeholder characters, so their contents are checked before trimming or treating values as missing.

### Type decisions based on the column meanings

| Fields | How they are stored | Treatment and reason |
|---|---|---|
| `Mmsi`, `ImoNumber` | Unsigned integers | Keep as identifiers or coded values. Do not treat them as quantities to average. |
| `MessageType`, `ShipType`, status, fix type, ETA parts | Integers | Keep the source codes and add readable or interpreted fields where useful. A code is not a measurement. |
| `Date` / `time` | Timestamp without timezone label | Keep the original timestamp and add a UTC-labelled version based on the source convention, without shifting the clock. |
| Latitude, longitude, speed, course, weather and ocean fields | Floating-point values | Retain their precision and check their plausible ranges and units. |
| Callsign, vessel name, destination | Text | Keep the source text; create trimmed versions and handle documented `@` placeholders in derived fields. |
| `geometry`, `id_chunk` | Binary geometry / list of text values | Preserve these source fields. Use the numeric coordinates for location checks and comparisons. |
| Ship-type lookup `Code` | CSV-inferred numeric field | Check its values and uniqueness, then convert to a nullable integer for a reliable code match. |

A column is not converted just because its storage type looks inconvenient. Its description and observed values help determine whether it is an identifier, code, measurement, or text.

## 2. Check missing values and consider why they are missing

A blank value does not always mean a bad observation. An AIS message may simply not carry every field, and an environmental grid may have locations where the model provides no value. Missing values are counted across all six files, then interpreted using the report type or grid location. This avoids deleting useful rows because one field is blank or filling a blank with an assumed value.

In [3]:
# Sum row-group null counts so the full table need not be materialized for this check.
null_rows = []
for name, parquet_file in PARQUET.items():
    # Initialize a counter for each top-level field in this dataset.
    counts = {field.name: 0 for field in parquet_file.schema_arrow}
    for row_group_index in range(parquet_file.metadata.num_row_groups):
        # Inspect metadata for one row group at a time.
        row_group = parquet_file.metadata.row_group(row_group_index)
        for column_index in range(row_group.num_columns):
            # Add null counts where Parquet exposes statistics for that leaf column.
            column = row_group.column(column_index)
            stats = column.statistics
            if stats is not None and column.path_in_schema in counts:
                counts[column.path_in_schema] += stats.null_count
    # Parquet stores nested list elements under a leaf path; count list-row nulls explicitly.
    if name == "AIS_POS":
        # Request one column only, so it is at position zero in each returned batch.
        # Streaming that column counts top-level null lists without loading all AIS fields.
        counts["id_chunk"] = sum(
            batch.column(0).null_count
            for batch in parquet_file.iter_batches(
                batch_size=500_000, columns=["id_chunk"])
        )
    # Convert absolute null totals into percentages for cross-dataset comparison.
    for field, null_count in counts.items():
        total = parquet_file.metadata.num_rows
        null_rows.append({"dataset": name, "field": field, "nulls": null_count,
                          "null_percent": 100 * null_count / total if total else np.nan})
null_profile = pd.DataFrame(null_rows)
# Display fields with any reported nulls; a zero-row result would be informative too.
display(null_profile.loc[null_profile["nulls"].gt(0)].sort_values(
    ["dataset", "null_percent"], ascending=[True, False]))

,dataset,field,nulls,null_percent
12,AIS_POS,chunk_folder,18973660,99.786639
13,AIS_POS,id_chunk,18973660,99.786639
10,AIS_POS,RateOfTurn,13611536,71.586053
4,AIS_POS,NavigationStatus,9802517,51.553587
11,AIS_POS,TrueHeadingDegrees,3809019,20.032466
8,AIS_POS,CourseOverGroundDegrees,3711937,19.521891
9,AIS_POS,SpeedOverGround,53172,0.279643
18,AIS_SPEC,ImoNumber,9218924,67.996159
26,AIS_SPEC,Draught10thMetres,9218924,67.996159
27,AIS_SPEC,Destination,9218924,67.996159


### What the missing-value counts show

The files do not have one common missing-data problem. In AIS_POS, navigation status, course, speed, rate of turn, and heading have different numbers of blanks; some fields are not present in every type of AIS position report. In AIS_SPEC, IMO number, draught, destination, and ETA parts are often blank, and vessel name is blank slightly less often; these are consistent with the different content carried by static and voyage reports.

The environmental files show a different pattern: CMEMS_PHY has 402,038 missing values per variable at the same set of 182 grid locations. CMEMS_WAV has stable location-based gaps (274 locations for most fields and 115 for directions), and `VMXL` is blank throughout. ERA5 has no source nulls. The fact that environmental gaps recur at the same locations suggests a coverage or land/coastal mask, rather than values randomly going missing over time. The grid rows and blanks are retained, and AIS fields are not filled globally.

## 3. AIS_POS: understand the position reports before changing them

AIS_POS can be used to follow vessel movement: it includes a vessel identifier, report time, position, and movement or navigation fields. The checks cover latitude and longitude limits, negative speed values, special course, heading, and rate-of-turn codes, and blanks by message type.

`(Mmsi, Date)` is tested as a possible event identifier, but is not assumed to be unique. Reports with the same vessel and time are compared across all fields, including the source-record list, before deciding whether one is a repeated copy.

In [4]:
# Read only movement fields needed for coordinate, speed, and sentinel checks.
pos_fields = ["Mmsi", "Date", "MessageType", "Latitude", "Longitude", "SpeedOverGround",
              "CourseOverGroundDegrees", "TrueHeadingDegrees", "RateOfTurn", "NavigationStatus",
              "PositionAccuracy", "Source", "chunk_folder", "id_chunk"]
pos = pd.read_parquet(FILES["AIS_POS"], columns=pos_fields)

# Check coordinate ranges against geographic latitude/longitude limits.
position_checks = {
    "rows": len(pos),
    "latitude_outside_minus90_to90": int((~pos["Latitude"].between(-90, 90)).sum()),
    "longitude_outside_minus180_to180": int((~pos["Longitude"].between(-180, 180)).sum()),
    "negative_sog": int(pos["SpeedOverGround"].lt(0).sum()),
    "sog_at_or_above_102_19": int(pos["SpeedOverGround"].ge(102.19).sum()),
    "message27_sog_63_unavailable": int((pos["MessageType"].eq(27) & pos["SpeedOverGround"].eq(63)).sum()),
    "cog_non_null_outside_0_359": int((pos["CourseOverGroundDegrees"].notna() &
        ~pos["CourseOverGroundDegrees"].between(0, 359)).sum()),
    "heading_non_null_outside_0_359": int((pos["TrueHeadingDegrees"].notna() &
        ~pos["TrueHeadingDegrees"].between(0, 359)).sum()),
    "rot_no_information_minus128": int(pos["RateOfTurn"].eq(-128).sum()),
}
print(json.dumps(position_checks, indent=2))

# Profile null rates within message type to distinguish structural missingness from bad capture.
pos_missing_by_type = pos.groupby("MessageType")[
    ["NavigationStatus", "SpeedOverGround", "CourseOverGroundDegrees", "TrueHeadingDegrees", "RateOfTurn"]
].agg(lambda column: column.isna().mean() * 100)
print("Null percentage within AIS message type:")
display(pos_missing_by_type.round(2))

# Identify repeated event keys while retaining all rows in repeated groups for comparison.
position_key = ["Mmsi", "Date"]
repeated_pos = pos.loc[pos.duplicated(position_key, keep=False)].copy()
print(f"Repeated-key rows: {len(repeated_pos):,}; repeated key groups: "
      f"{repeated_pos.groupby(position_key, dropna=False).ngroups:,}")

# Convert the nested list field to tuples so equality comparisons are well-defined.
repeated_pos["id_chunk_compare"] = repeated_pos["id_chunk"].map(
    lambda value: tuple(value) if isinstance(value, (list, tuple, np.ndarray)) else value)
# Compare every field inside each repeated key; nunique > 1 signals a conflicting payload.
comparison_fields = [field for field in repeated_pos.columns if field != "id_chunk"] + ["id_chunk_compare"]
variation_by_field = repeated_pos.groupby(position_key, dropna=False)[comparison_fields].nunique(dropna=False)
print("Repeated keys with field variation:")
display(variation_by_field.gt(1).sum().rename("groups_with_variation").to_frame())

# Count common encoded values separately so a special AIS code is not mistaken for a degree.
ais_position_code_counts = {
    "COG_equals_511": int(pos["CourseOverGroundDegrees"].eq(511).sum()),
    "COG_equals_360": int(pos["CourseOverGroundDegrees"].eq(360).sum()),
    "heading_equals_511": int(pos["TrueHeadingDegrees"].eq(511).sum()),
    "ROT_equals_minus128": int(pos["RateOfTurn"].eq(-128).sum()),
    "ROT_equals_minus127": int(pos["RateOfTurn"].eq(-127).sum()),
    "ROT_equals_plus127": int(pos["RateOfTurn"].eq(127).sum()),
}
print("AIS position encoded-value counts:", json.dumps(ais_position_code_counts, indent=2))
# Release large position frames before the next full-size AIS audit to limit notebook memory use.
del pos, repeated_pos, variation_by_field


{
  "rows": 19014229,
  "latitude_outside_minus90_to90": 0,
  "longitude_outside_minus180_to180": 0,
  "negative_sog": 0,
  "sog_at_or_above_102_19": 4,
  "message27_sog_63_unavailable": 22106,
  "cog_non_null_outside_0_359": 480745,
  "heading_non_null_outside_0_359": 10166008,
  "rot_no_information_minus128": 1273875
}
Null percentage within AIS message type:


,NavigationStatus,SpeedOverGround,CourseOverGroundDegrees,TrueHeadingDegrees,RateOfTurn
MessageType,,,,,
1,0.0,0.18,6.06,0.0,0.0
2,0.0,0.00,1.14,0.0,0.0
3,0.0,0.07,9.29,0.0,0.0
18,100.0,0.47,33.87,0.0,100.0
19,100.0,0.00,0.16,0.0,100.0
27,0.0,0.00,0.00,100.0,100.0


Repeated-key rows: 81,000; repeated key groups: 40,447
Repeated keys with field variation:


,groups_with_variation
Mmsi,0
Date,0
MessageType,0
Latitude,0
Longitude,0
SpeedOverGround,0
CourseOverGroundDegrees,0
TrueHeadingDegrees,0
RateOfTurn,0
NavigationStatus,0


AIS position encoded-value counts: {
  "COG_equals_511": 456100,
  "COG_equals_360": 108,
  "heading_equals_511": 10164104,
  "ROT_equals_minus128": 1273875,
  "ROT_equals_minus127": 183648,
  "ROT_equals_plus127": 180512
}


### AIS_POS observations

There are 40,553 rows beyond the first record for repeated `(Mmsi, Date)` pairs, across 40,447 repeated pairs; 106 pairs appear three times. A comparison of all fields, including the source-record list, shows that the repeated records match exactly. This supports removing extra copies from the cleaned file.

The latitude and longitude values stay within their allowed geographic ranges, and there are no negative speeds. Four SOG values appear as `102.199997`, the float32 representation of the AIS upper-censored code 102.2 knots or higher. There is a second message-specific code: in long-range AIS Message Type 27, SOG value 63 means unavailable, not a measured 63 knots. The audit checks both rules; the cleaned speed field leaves both encoded values blank while preserving their source values and separate flags. (See the [US Coast Guard Message 27 specification](https://www.navcen.uscg.gov/sites/default/files/pdf/AIS/AIS_MSG27.pdf).) Course or heading outside 0–359 and rate of turn `-128` are handled as special or unusable codes, not clipped into ordinary measurements.

### AIS_POS cleaning decision and reason

Repeated vessel-and-time records are identical across every field, so only the extra copies are removed from the derived file. If positions or other details differed, the repeated key alone would not justify deleting a report. Missing navigation fields are retained because message type helps explain why they are absent; no assumed values are added.

For course, heading, rate of turn, and SOG, Original source values are preserved, with cleaned fields or flags added alongside them. This keeps the received values visible alongside their prepared versions.

## 4. AIS_SPEC: read vessel and voyage details as reported

AIS_SPEC holds static and voyage-related reports: vessel identity and dimensions, draught, ship type, destination, and estimated time of arrival. These reports can repeat, and a vessel can also send different information at the same recorded time. `(Mmsi, Date, MessageType)` is tested as a possible unique identifier. Full records are compared, and source descriptions are checked for code values and text placeholders before removing or converting anything.

In [5]:
# Load AIS_SPEC to inspect repeated reports, code values, and text placeholders.
spec = pd.read_parquet(FILES["AIS_SPEC"])
spec_key = ["Mmsi", "Date", "MessageType"]
repeated_spec = spec.loc[spec.duplicated(spec_key, keep=False)]
# Count extra rows by the candidate key, then count rows that are exact copies across every column.
repeated_key_extras = int(spec.duplicated(spec_key, keep="first").sum())
exact_duplicate_extras = int(spec.duplicated(keep="first").sum())
print({"rows_on_repeated_candidate_keys": len(repeated_spec),
       "repeated_candidate_key_extra_rows": repeated_key_extras,
       "exact_full_row_duplicate_extra_rows": exact_duplicate_extras})

# Check whether reports sharing the same vessel, time, and message type actually disagree in any field.
if len(repeated_spec):
    payload_columns = list(spec.columns)
    payload_variation = repeated_spec.groupby(spec_key, dropna=False)[payload_columns].nunique(dropna=False)
    conflicting_groups = payload_variation.gt(1).any(axis=1)
    print("Repeated-key groups with differing payload fields:", int(conflicting_groups.sum()))
    print("Fields most often different within a repeated key:")
    display(payload_variation.gt(1).sum().sort_values(ascending=False).head(10).to_frame("groups"))

# Compare IMO values with the ranges described for this AIS field before calling large values invalid.
imo = spec["ImoNumber"]
imo_profile = {
    "unavailable_code_zero": int(imo.eq(0).sum()),
    "unused_1_to_999999": int(imo.between(1, 999_999).sum()),
    "standard_imo_number_1000000_to_9999999": int(imo.between(1_000_000, 9_999_999).sum()),
    "official_flag_state_code_10000000_to_1073741823": int(imo.between(10_000_000, 1_073_741_823).sum()),
    "above_documented_upper_code": int(imo.gt(1_073_741_823).sum()),
}
print(json.dumps(imo_profile, indent=2))

# Find text made only of @ signs, which the AIS description uses as an unavailable placeholder.
for field in ["CallSign", "VesselName", "Destination"]:
    text_values = spec[field].dropna().astype("string")
    print(field, {"all_at_sign_sentinels": int(text_values.str.strip().str.fullmatch(r"@+", na=False).sum()),
                  "empty_after_trim": int(text_values.str.strip().eq("").sum())})

# Count draught codes before treating the field as an exact measurement.
draught_code_counts = {
    "unavailable_code_0": int(spec["Draught10thMetres"].eq(0).sum()),
    "exact_values_1_to_254": int(spec["Draught10thMetres"].between(1, 254).sum()),
    "upper_limit_code_255": int(spec["Draught10thMetres"].eq(255).sum()),
    "outside_documented_codes": int((~spec["Draught10thMetres"].between(0, 255)).sum()),
}
print("Draught code counts:", json.dumps(draught_code_counts, indent=2))

# Review dimension and ETA values so special codes are not mistaken for real measurements.
spec_code_summary = spec[["DimensionToBow", "DimensionToStern", "DimensionToPort",
                          "DimensionToStarboard", "Draught10thMetres", "EtaMonth", "EtaDay",
                          "EtaHour", "EtaMinute", "PositionFixType"]].describe().T
# Include actual minima/maxima; AIS's top dimension values are censored codes, not ordinary outliers.
display(spec_code_summary[["count", "mean", "std", "min", "max"]])

{'rows_on_repeated_candidate_keys': 21269, 'repeated_candidate_key_extra_rows': 10636, 'exact_full_row_duplicate_extra_rows': 10601}
Repeated-key groups with differing payload fields: 35
Fields most often different within a repeated key:


,groups
Source,33
ShipType,3
DimensionToStern,2
DimensionToStarboard,2
DimensionToPort,2
DimensionToBow,2
PositionFixType,2
VesselName,2
CallSign,2
Destination,1


{
  "unavailable_code_zero": 1025321,
  "unused_1_to_999999": 14301,
  "standard_imo_number_1000000_to_9999999": 3258786,
  "official_flag_state_code_10000000_to_1073741823": 40675,
  "above_documented_upper_code": 0
}
CallSign {'all_at_sign_sentinels': 299150, 'empty_after_trim': 0}
VesselName {'all_at_sign_sentinels': 2133, 'empty_after_trim': 0}
Destination {'all_at_sign_sentinels': 448430, 'empty_after_trim': 0}
Draught code counts: {
  "unavailable_code_0": 524279,
  "exact_values_1_to_254": 3814625,
  "upper_limit_code_255": 179,
  "outside_documented_codes": 9218924
}


,count,mean,std,min,max
DimensionToBow,13557902.0,26.718404,46.792497,0.0,511.0
DimensionToStern,13557902.0,14.188809,32.150237,0.0,511.0
DimensionToPort,13557902.0,4.142872,5.335214,0.0,63.0
DimensionToStarboard,13557902.0,4.087918,5.329054,0.0,63.0
Draught10thMetres,4339083.0,50.971599,36.458305,0.0,255.0
EtaMonth,4339083.0,6.017633,3.500798,0.0,13.0
EtaDay,4339083.0,12.484105,10.197297,0.0,31.0
EtaHour,4339083.0,12.464482,7.922655,0.0,25.0
EtaMinute,4339083.0,14.482363,22.653284,0.0,61.0
PositionFixType,13557902.0,0.960353,3.072359,0.0,15.0


### AIS_SPEC observations

There are 10,636 extra rows when records are grouped by `(Mmsi, Date, MessageType)`. Of these, 10,601 are exact copies of an earlier full row. The remaining repeated-key records contain different information, so both are retained rather than choosing one arbitrarily.

Calls sign, vessel name, and destination include padded text and strings made entirely of `@` characters. These are not useful names or destinations, so the cleaned copy trims the text and treats the documented placeholder as unavailable. Dimension, draught, and ETA values were compared with the AIS field descriptions. Draught code 255 means 25.5 m or greater, so it is an upper-limit value rather than an exact draught. Some large `ImoNumber` values are documented flag-state codes; they are not automatically errors just because they exceed a normal seven-digit IMO number.

### AIS_SPEC and ship-type lookup: cleaning decisions

Only exact duplicate rows are removed. Where the possible unique identifier repeats but the details differ, both reports are kept because the file does not show which one is more reliable. Each ETA part is interpreted separately so an unavailable month, day, hour, or minute does not become a real time. Draught is recorded in tenths of a metre: codes 1–254 convert to metres, code 0 means unavailable, and code 255 means 25.5 m or greater. The upper-limit code is flagged and left out of the exact draught field. Text fields are trimmed and all-`@` placeholders are marked unavailable in new fields; received values are preserved.

The ship-type CSV supplies names for numeric ship-type codes. It needs to be read with its actual semicolon separator. Before adding names, the lookup is checked for unique codes and labels for the codes found in AIS_SPEC. Code 0 is explicitly “Not available”; it is a category with a label, not a missing lookup.

In [6]:
# Read the ship-type lookup using the semicolon separator used in the supplied file.
ship_types = pd.read_csv(FILES["AIS_ShipTypes"], sep=";")
# Make lookup codes comparable with AIS_SPEC, then check duplicates and whether observed codes have labels.
ship_types["Code"] = pd.to_numeric(ship_types["Code"], errors="raise").astype("Int64")
observed_ship_codes = set(spec["ShipType"].dropna().astype(int).unique())
lookup_codes = set(ship_types["Code"].dropna().astype(int).unique())
lookup_audit = {
    "lookup_rows": len(ship_types),
    "null_cells": int(ship_types.isna().sum().sum()),
    "duplicate_codes": int(ship_types["Code"].duplicated().sum()),
    "observed_ship_type_codes": len(observed_ship_codes),
    "observed_codes_without_lookup": sorted(observed_ship_codes - lookup_codes),
    "lookup_codes_not_observed": len(lookup_codes - observed_ship_codes),
}
print(json.dumps(lookup_audit, indent=2))
# Free memory after finishing the AIS_SPEC checks.
del spec, repeated_spec, payload_variation


{
  "lookup_rows": 256,
  "null_cells": 0,
  "duplicate_codes": 0,
  "observed_ship_type_codes": 95,
  "observed_codes_without_lookup": [],
  "lookup_codes_not_observed": 161
}


### Ship-type lookup observations

The lookup has 256 populated, unique codes. All 95 ship-type codes observed in this AIS_SPEC extract have a matching lookup entry. The other 161 lookup codes simply do not appear in this particular extract. Code 0 maps to “Not available,” so it remains a labelled category.

## 5. Environmental files: understand the grids and their coverage

Unlike AIS, an environmental row describes conditions rather than a ship report. It describes a condition at a particular time and grid location; CMEMS_PHY also includes depth. Before comparing these files with vessel positions, it is necessary to know whether each time-and-location combination is unique, how often the grid is updated, how many locations are present at each time, and whether missing values are tied to particular places.

Ranges and units are also checked against the data descriptions. This is important because a value can look unusual simply because the local file uses a different scale or unit than expected.

In [7]:
# List each grid's location/time columns and the environmental measurements to inspect.
ENVIRONMENTAL = {
    "ERA5": {"keys": ["time", "latitude", "longitude"],
             "values": ["sp", "msl", "tcc", "u10", "v10", "t2m", "d2m"]},
    "CMEMS_PHY": {"keys": ["time", "depth", "latitude", "longitude"],
                  "values": ["so", "thetao", "uo", "vo", "zos"]},
    "CMEMS_WAV": {"keys": ["time", "latitude", "longitude"],
                  "values": ["VCMX", "VHM0", "VHM0_SW1", "VHM0_SW2", "VHM0_WW", "VMDR",
                             "VMDR_SW1", "VMDR_SW2", "VMDR_WW", "VMXL", "VPED", "VSDX",
                             "VSDY", "VTM01_SW1", "VTM01_SW2", "VTM01_WW", "VTM02", "VTM10", "VTPK"]},
}

environment_summaries = {}
for name, spec_info in ENVIRONMENTAL.items():
    # Load one environmental file at a time to check its records, coverage, and value ranges.
    grid = pd.read_parquet(FILES[name])
    keys = spec_info["keys"]
    values = spec_info["values"]
    # Check that a time and grid location identify only one row, to avoid duplicating records in a later match.
    duplicate_rows = int(grid.duplicated(keys, keep=False).sum())
    rows_per_time = grid.groupby("time", sort=True).size()
    time_steps = grid["time"].drop_duplicates().sort_values().diff().dropna()
    # Record how many locations occur at each time and how far apart the timestamps are.
    cadence_counts = {str(step): int(count) for step, count in time_steps.value_counts().head(5).items()}
    environment_summaries[name] = {
        "rows": len(grid), "duplicate_key_rows": duplicate_rows,
        "timestamps": int(grid["time"].nunique()),
        "cells_per_time_min": int(rows_per_time.min()),
        "cells_per_time_max": int(rows_per_time.max()),
        "time_step_examples": cadence_counts,
    }
    print(f"\n{name}: key={keys}")
    print(json.dumps(environment_summaries[name], indent=2))
    # Summarize blanks and value ranges to see where data are absent and whether values need investigation.
    display(grid[values].agg(["count", "min", "median", "max"]).T)
    # Check whether each location is always blank or only blank at some times.
    for field in values:
        if grid[field].isna().any():
            cell_missing = grid.groupby(["latitude", "longitude"], sort=False)[field].agg(
                total="size", missing="count")
            cell_missing["missing"] = cell_missing["total"] - cell_missing["missing"]
            always_masked = int(cell_missing["missing"].eq(cell_missing["total"]).sum())
            intermittent = int((cell_missing["missing"].gt(0) &
                                cell_missing["missing"].lt(cell_missing["total"])).sum())
            print(f"{field}: always-masked cells={always_masked:,}; "
                  f"intermittently missing cells={intermittent:,}; null rows={int(grid[field].isna().sum()):,}")
# Release the last grid dataframe before writing the larger AIS clean outputs.
del grid, cell_missing, rows_per_time, time_steps



ERA5: key=['time', 'latitude', 'longitude']
{
  "rows": 537510,
  "duplicate_key_rows": 0,
  "timestamps": 2185,
  "cells_per_time_min": 246,
  "cells_per_time_max": 246,
  "time_step_examples": {
    "0 days 01:00:00": 2184
  }
}


,count,min,median,max
sp,537510.0,973.195618,1013.809387,1027.343750
msl,537510.0,983.940613,1015.054871,1027.850830
tcc,537510.0,0.000000,69.247589,100.000618
u10,537510.0,-11.316105,4.027392,19.413616
v10,537510.0,-12.149607,1.098410,18.558441
t2m,537510.0,8.168365,17.085114,30.083893
d2m,537510.0,6.243805,14.634430,22.027985



CMEMS_PHY: key=['time', 'depth', 'latitude', 'longitude']
{
  "rows": 5007803,
  "duplicate_key_rows": 0,
  "timestamps": 2209,
  "cells_per_time_min": 2267,
  "cells_per_time_max": 2267,
  "time_step_examples": {
    "0 days 01:00:00": 2208
  }
}


,count,min,median,max
so,4605765.0,31.390125,34.841118,35.738358
thetao,4605765.0,10.593507,17.439426,23.311535
uo,4605765.0,-0.563283,0.039770,0.907670
vo,4605765.0,-0.567463,-0.001924,1.077356
zos,4605765.0,-0.941837,-0.313222,0.424946


so: always-masked cells=182; intermittently missing cells=0; null rows=402,038
thetao: always-masked cells=182; intermittently missing cells=0; null rows=402,038
uo: always-masked cells=182; intermittently missing cells=0; null rows=402,038
vo: always-masked cells=182; intermittently missing cells=0; null rows=402,038
zos: always-masked cells=182; intermittently missing cells=0; null rows=402,038

CMEMS_WAV: key=['time', 'latitude', 'longitude']
{
  "rows": 1670779,
  "duplicate_key_rows": 0,
  "timestamps": 737,
  "cells_per_time_min": 2267,
  "cells_per_time_max": 2267,
  "time_step_examples": {
    "0 days 03:00:00": 736
  }
}


,count,min,median,max
VCMX,1468841.0,0.040000,2.150000,13.53
VHM0,1468841.0,0.020000,1.130000,7.27
VHM0_SW1,1468841.0,0.010000,0.730000,5.81
VHM0_SW2,1468841.0,0.000000,0.170000,3.42
VHM0_WW,1468841.0,0.000000,0.500000,7.14
VMDR,1586024.0,0.000004,260.540009,360.00
VMDR_SW1,1586024.0,0.000004,265.529999,360.00
VMDR_SW2,1586024.0,0.000004,242.729996,360.00
VMDR_WW,1586024.0,0.000004,239.919998,360.00
VMXL,0.0,NaN,NaN,NaN


VCMX: always-masked cells=274; intermittently missing cells=0; null rows=201,938
VHM0: always-masked cells=274; intermittently missing cells=0; null rows=201,938
VHM0_SW1: always-masked cells=274; intermittently missing cells=0; null rows=201,938
VHM0_SW2: always-masked cells=274; intermittently missing cells=0; null rows=201,938
VHM0_WW: always-masked cells=274; intermittently missing cells=0; null rows=201,938
VMDR: always-masked cells=115; intermittently missing cells=0; null rows=84,755
VMDR_SW1: always-masked cells=115; intermittently missing cells=0; null rows=84,755
VMDR_SW2: always-masked cells=115; intermittently missing cells=0; null rows=84,755
VMDR_WW: always-masked cells=115; intermittently missing cells=0; null rows=84,755
VMXL: always-masked cells=2,267; intermittently missing cells=0; null rows=1,670,779
VPED: always-masked cells=115; intermittently missing cells=0; null rows=84,755
VSDX: always-masked cells=274; intermittently missing cells=0; null rows=201,938
VSDY: a

### Environmental-grid observations

The time-and-location keys do not repeat, and the grids have regular time steps. ERA5 contains 2,185 hourly timestamps with 246 locations at each timestamp. CMEMS_PHY contains 2,209 hourly timestamps with 2,267 locations; CMEMS_WAV contains 737 timestamps spaced three hours apart, also with 2,267 locations. The missing values line up with fixed locations, so the grid and its coverage pattern are retained instead of filling the gaps.

One scale difference needs attention: local ERA5 `tcc` values run from 0 to about 100, while the MMDEC description gives cloud cover as a fraction from 0 to 1. The source column is preserved, and a converted fraction column is added for later use.

### Cleaning decisions for the three environmental files

- **ERA5 weather:** The time/location key is unique, each hour has 246 grid locations, and there are no source blanks. The MMDEC description gives pressure in hPa, cloud cover as a 0–1 fraction, temperature in °C, and wind components in m/s. In this local file `tcc` is on a 0–100 scale and reaches 100.00062. The source value is preserved, and a fraction field is added by dividing by 100; the tiny amount above 100 is consistent with storage precision.
- **CMEMS_PHY ocean conditions:** The same fields are blank at the same 182 grid locations throughout the period. Those rows and blanks are retained because they show where the grid has no value; filling them would invent ocean measurements.
- **CMEMS_WAV wave conditions:** The missing locations differ by field but stay fixed over time. `VMXL` is blank in every row and cannot contribute information, so the cleaned copy omits that unusable field while retaining all rows. Direction values are expressed on a circle, where 360° is the same direction as 0°; the derived direction fields use the 0–359° convention.

The timestamps are stored without a timezone label. The cleaned copies add a UTC-labelled version, based on the AIS and source-product conventions, without changing the recorded clock values. Time alignment and units must be confirmed before matching environmental conditions to vessel positions. The data inventory documents field meanings and source relationships.

## 6. Apply the cleaning rules to separate copies

Each cleaning rule follows from the checks above. The cleaning functions are run one file at a time, making the source of each output clear. The original downloads remain unchanged. The functions keep source-coded values where possible and add cleaned or interpreted fields alongside them, allowing later users to trace a value back to what the source supplied.

In [8]:
# Load function definitions without invoking the script's all-dataset main routine.
from runpy import run_path
cleaner = run_path(str(ROOT / "scripts" / "clean_mmdec.py"), run_name="cleaning_functions")

# Clean AIS_POS using the evidence that repeated vessel-time records were exact copies.
ais_pos_result = cleaner["clean_ais_positions"]()
print(json.dumps(ais_pos_result, indent=2))

{
  "source_rows": 19014229,
  "output_rows": 18973676,
  "removed_confirmed_duplicate_rows": 40553,
  "cog_non_null_outside_0_359": 480659,
  "heading_non_null_outside_0_359": 10138994,
  "rot_code_minus_128": 1270642,
  "sog_at_or_above_102_2_flagged": 4,
  "message_27_sog_63_unavailable": 22106
}


In [9]:
# Clean AIS_SPEC by removing exact copies while keeping same-key reports with different details.
ais_spec_result = cleaner["clean_ais_specifications"]()
print(json.dumps(ais_spec_result, indent=2))

{
  "source_rows": 13558007,
  "output_rows": 13547406,
  "removed_exact_whole_row_duplicates": 10601,
  "imo_unavailable_code_rows": 1024540,
  "imo_standard_number_rows": 3256209,
  "imo_flag_state_number_rows": 40639,
  "imo_reserved_or_invalid_code_rows": 14292,
  "draught_unavailable_code_rows": 523846,
  "draught_upper_limit_code_rows": 179,
  "eta_month_missing_or_invalid": 10082073,
  "eta_day_missing_or_invalid": 10069075,
  "eta_hour_missing_or_invalid": 9836246,
  "eta_minute_missing_or_invalid": 9836515,
  "unmapped_ship_type_rows": 0
}


In [10]:
# Save the ship-type lookup after confirming its codes are unique and cover the observed vessel codes.
ship_type_result = cleaner["clean_ship_type_lookup"]()
print(json.dumps(ship_type_result, indent=2))

{
  "rows": 256,
  "null_cells": 0,
  "duplicate_codes": 0
}


In [11]:
# Prepare an ERA5 copy with a UTC-labelled time and cloud cover expressed as a fraction.
era5_result = cleaner["copy_environmental_file"]("Dataset_ERA5.parquet", "era5_clean.parquet")
print(json.dumps(era5_result, indent=2))

{
  "rows": 537510,
  "source_null_counts": {
    "time": 0,
    "latitude": 0,
    "longitude": 0,
    "sp": 0,
    "msl": 0,
    "tcc": 0,
    "u10": 0,
    "v10": 0,
    "t2m": 0,
    "d2m": 0,
    "geometry": 0
  },
  "tcc_source_min": 0.0,
  "tcc_source_max": 100.00061798095703,
  "tcc_fraction_clean_min": 0.0,
  "tcc_fraction_clean_max": 1.0
}


In [12]:
# Prepare the ocean-physics copy while keeping all grid locations and their recurring blanks.
phy_result = cleaner["copy_environmental_file"]("Dataset_CMEMS_PHY.parquet", "cmems_phy_clean.parquet")
print(json.dumps(phy_result, indent=2))

{
  "rows": 5007803,
  "source_null_counts": {
    "time": 0,
    "depth": 0,
    "latitude": 0,
    "longitude": 0,
    "so": 402038,
    "thetao": 402038,
    "uo": 402038,
    "vo": 402038,
    "zos": 402038,
    "geometry": 0
  }
}


In [13]:
# Prepare the wave copy, keep its coverage pattern, standardize direction values, and omit the always-blank field.
wav_result = cleaner["copy_environmental_file"](
    "Dataset_CMEMS_WAV.parquet", "cmems_wav_clean.parquet", all_null_columns=("VMXL",))
print(json.dumps(wav_result, indent=2))

{
  "rows": 1670779,
  "source_null_counts": {
    "time": 0,
    "latitude": 0,
    "longitude": 0,
    "VCMX": 201938,
    "VHM0": 201938,
    "VHM0_SW1": 201938,
    "VHM0_SW2": 201938,
    "VHM0_WW": 201938,
    "VMDR": 84755,
    "VMDR_SW1": 84755,
    "VMDR_SW2": 84755,
    "VMDR_WW": 84755,
    "VMXL": 1670779,
    "VPED": 84755,
    "VSDX": 201938,
    "VSDY": 201938,
    "VTM01_SW1": 201938,
    "VTM01_SW2": 201938,
    "VTM01_WW": 201938,
    "VTM02": 201938,
    "VTM10": 201938,
    "VTPK": 201938,
    "geometry": 0
  }
}


In [14]:
# Combine the six dataset-level results produced in the cells above.
notebook_cleaning_summary = {
    "AIS_POS": ais_pos_result,
    "AIS_SPEC": ais_spec_result,
    "AIS_ShipTypes": ship_type_result,
    "ERA5": era5_result,
    "CMEMS_PHY": phy_result,
    "CMEMS_WAV": wav_result,
}
# Save this notebook run's actual findings so the JSON matches the displayed cell outputs.
summary_path = ROOT / "data" / "processed" / "cleaning_summary.json"
summary_path.write_text(json.dumps(notebook_cleaning_summary, indent=2))
print(f"Updated run summary: {summary_path}")

Updated run summary: /Users/midhunpraj/Documents/Maritime Traffic & Environmental Performance Analytics/data/processed/cleaning_summary.json


## 7. Check the files produced by cleaning

After each cleaned file is written, its Parquet metadata is reopened and the number of rows and columns is compared with the source. This shows, for example, where exact duplicate removal changed the row count and where added interpretation fields increased the number of columns. The saved run summary is also checked. These checks confirm that the outputs were written as expected; they do not yet tell us anything about maritime patterns.

In [15]:
# List the cleaned outputs and source row counts for a before-and-after comparison.
EXPECTED_OUTPUTS = {
    "AIS_POS": ("ais_positions_clean.parquet", PARQUET["AIS_POS"].metadata.num_rows),
    "AIS_SPEC": ("ais_specifications_clean.parquet", PARQUET["AIS_SPEC"].metadata.num_rows),
    "ERA5": ("era5_clean.parquet", PARQUET["ERA5"].metadata.num_rows),
    "CMEMS_PHY": ("cmems_phy_clean.parquet", PARQUET["CMEMS_PHY"].metadata.num_rows),
    "CMEMS_WAV": ("cmems_wav_clean.parquet", PARQUET["CMEMS_WAV"].metadata.num_rows),
}
verification_rows = []
for name, (filename, input_rows) in EXPECTED_OUTPUTS.items():
    # Compare source and cleaned row counts from file metadata without loading large datasets.
    output_file = pq.ParquetFile(ROOT / "data" / "processed" / filename)
    verification_rows.append({"dataset": name, "input_rows": input_rows,
                              "output_rows": output_file.metadata.num_rows,
                              "columns_after_cleaning": len(output_file.schema_arrow)})
# Show row and column counts to make the effect of cleaning visible.
display(pd.DataFrame(verification_rows))

# Reopen the saved run summary to confirm the cleaning results were recorded.
with open(ROOT / "data" / "processed" / "cleaning_summary.json") as summary_file:
    cleaning_summary = json.load(summary_file)
print("Run summary contains:", list(cleaning_summary))

,dataset,input_rows,output_rows,columns_after_cleaning
0,AIS_POS,19014229,18973676,22
1,AIS_SPEC,13558007,13547406,36
2,ERA5,537510,537510,13
3,CMEMS_PHY,5007803,5007803,11
4,CMEMS_WAV,1670779,1670779,28


Run summary contains: ['AIS_POS', 'AIS_SPEC', 'AIS_ShipTypes', 'ERA5', 'CMEMS_PHY', 'CMEMS_WAV']


## 8. Screen for possible jumps between AIS positions

As a final data-quality screen, each vessel's position is compared with its next recorded position to estimate the speed needed to travel between them. A very high implied speed can point to a wrong coordinate, a timing issue, an MMSI reused by another vessel, or a fast vessel. It is a reason to inspect a segment, not proof that the report is wrong.

The script limits this first screen to observations no more than six hours apart and lists segments above 40 knots for review. It does not delete or change any position. The threshold is a review aid, and any later analysis would need to account for vessel type and reporting gaps.

In [16]:
# Run the position-jump review and display its summary; it only flags candidates for inspection.
track_screen_namespace = run_path(str(ROOT / "scripts" / "screen_tracks.py"),
                                  run_name="track_screen_workflow")
with open(ROOT / "data" / "processed" / "trajectory_screen_summary.json") as track_file:
    track_summary = json.load(track_file)
print(json.dumps(track_summary, indent=2))

{
  "input_position_rows": 18973676,
  "positive_time_segments": 18948546,
  "segments_gap_over_six_hours": 186270,
  "short_segments_up_to_six_hours": 18762276,
  "short_segment_speed_quantiles_knots": {
    "0.5": 0.014880247758140949,
    "0.9": 11.044327684025987,
    "0.95": 13.370300419253393,
    "0.99": 17.918939169913266,
    "0.999": 23.7517769071528
  },
  "short_segments_over_30_knots": 5874,
  "short_segments_over_40_knots": 81,
  "short_segments_over_60_knots": 5,
  "distinct_mmsi_with_over_40_knots": 69,
  "short_segments_over_100_knots": 0,
  "max_short_segment_speed_knots": 71.76082928948284,
  "reported_sog_over_40_rows": 495,
  "high_sog_stationary_position_candidate_rows": 16,
  "high_sog_stationary_position_candidate_mmsi": 4,
  "high_sog_stationary_candidate_rule": "At least 3 measured SOG reports above 40 kn spanning at least 24 hours, all within 100 m of their mean position; review flag only.",
  "interpretation": "Movement and reported-speed inconsistencies are

## Ready to begin EDA

All six files have now been described; their stored types and observed values, missingness, and repeated records have been checked; and the cleaning choices and output checks have been recorded. Possible AIS track jumps are listed for review and have not been removed.

The next stage is to explore the cleaned data: first describe AIS coverage and reporting intervals, then look at ship types, speed and navigation-status distributions, and where vessels appear in the study area. The findings can then guide whether the data support a useful question involving environmental conditions. Time conventions and variable units must be confirmed before matching vessel reports to weather or ocean grids.